In [ ]:
import json
import time
import traceback
from pathlib import Path

import requests
from cryptography.fernet import Fernet

INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)


def _find_first(filename: str) -> Path | None:
    for path in INPUT_ROOT.rglob(filename):
        if path.is_file():
            return path
    return None


def load_config() -> dict:
    config_path = _find_first("config.json")
    if config_path is None:
        raise RuntimeError("config.json not found under /kaggle/input")
    return json.loads(config_path.read_text(encoding="utf-8"))


def load_secrets() -> dict:
    enc_path = _find_first("secrets.enc")
    key_path = _find_first("fernet.key")
    if enc_path is None or key_path is None:
        raise RuntimeError("secrets.enc/fernet.key not found under /kaggle/input")
    payload = Fernet(key_path.read_bytes().strip()).decrypt(enc_path.read_bytes())
    return json.loads(payload.decode("utf-8"))


def pick_secret(config: dict, secrets: dict) -> tuple[str, str]:
    names = [str(config.get("secret_env_var") or "").strip()]
    names.extend(str(item).strip() for item in (config.get("secret_env_aliases") or []))
    for name in names:
        if name and str(secrets.get(name) or "").strip():
            return name, str(secrets[name]).strip()
    raise RuntimeError(f"Secret not found in bundle for lookup order: {names}")


def extract_response_text(payload: dict) -> str:
    texts: list[str] = []
    for candidate in payload.get("candidates") or []:
        content = candidate.get("content") or {}
        for part in content.get("parts") or []:
            text = str(part.get("text") or "").strip()
            if text:
                texts.append(text)
    return "\n".join(texts).strip()


In [ ]:
config = load_config()
secrets = load_secrets()
secret_name, api_key = pick_secret(config, secrets)
model = str(config.get("model") or "models/gemma-3-27b-it").strip()
if not model.startswith("models/"):
    model = f"models/{model}"
prompt = str(config.get("prompt") or "Reply with exactly: OK")
max_output_tokens = max(1, int(config.get("max_output_tokens") or 8))
timeout_seconds = max(10, int(config.get("request_timeout_seconds") or 90))
endpoint = f"https://generativelanguage.googleapis.com/v1beta/{model}:generateContent?key={api_key}"

result = {
    "started_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "probe_name": str(config.get("probe_name") or "gemma-key2-probe"),
    "secret_env_var": secret_name,
    "model": model,
    "prompt_excerpt": prompt[:160],
}

try:
    response = requests.post(
        endpoint,
        headers={"Content-Type": "application/json"},
        json={
            "contents": [{"parts": [{"text": prompt}]}],
            "generationConfig": {"maxOutputTokens": max_output_tokens},
        },
        timeout=timeout_seconds,
    )
    result["status_code"] = int(response.status_code)
    result["ok"] = bool(response.ok)
    try:
        payload = response.json()
    except Exception:
        payload = {"raw_text": response.text[:4000]}
    result["response_excerpt"] = extract_response_text(payload)[:1000]
    result["error_excerpt"] = json.dumps(payload, ensure_ascii=False)[:2000]
except Exception as exc:
    result["ok"] = False
    result["exception_type"] = exc.__class__.__name__
    result["exception"] = str(exc)[:1000]
    result["traceback_excerpt"] = traceback.format_exc()[:4000]

result["finished_at"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
(WORK_DIR / "output.json").write_text(
    json.dumps(result, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(json.dumps({
    "ok": result.get("ok"),
    "status_code": result.get("status_code"),
    "model": result.get("model"),
    "secret_env_var": result.get("secret_env_var"),
    "response_excerpt": (result.get("response_excerpt") or result.get("error_excerpt") or "")[:240],
}, ensure_ascii=False))
